# Group 8 - Backbone experiment on Google Colab (EfficientNetV2-S / ConvNeXt-Tiny)

**This is an experiment, not the submitted notebook.** Guideline B (item 1.4) asks for a VGG, ResNet or Inception
backbone, and doc A fixes ResNet. Five ResNet-50 runs stayed at 85 - 86 % validation accuracy, so this notebook
checks how far two modern backbones get with **exactly the same data, split, metadata and training recipe**. The
result is evidence for the discussion with the lecturer; it does not replace `Group_8_Image_Recognition.ipynb`.

**How to run**

1. Upload `g8_data.zip` (about 1 GB: 30,926 images already resized to 384 x 384 + `g8_labels.csv` with the split)
   to Google Drive, either directly into *My Drive* or into a folder inside it (e.g. `MyDrive/Group_8/`).
2. Open this notebook in Colab: *Runtime -> Change runtime type -> T4 GPU* (or any GPU).
3. Choose `BACKBONE` in the settings cell, then *Runtime -> Run all* and allow access to Google Drive.
4. Keep the browser tab open. Results go to a `results/<backbone>_384/` folder next to the zip. If Colab disconnects, run all
   cells again: training continues from the last finished epoch.
5. Run it a second time with the other `BACKBONE`, then send the printed results (last cell) and
   `summary.csv` of both runs.

Expected time on a T4 with mixed precision: about 1 - 2 hours per backbone (the first epoch prints its duration;
10 epochs in total).

In [ ]:
import os
import math
import time
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from sklearn.metrics import classification_report

try:
    from google.colab import drive
    drive.mount("/content/drive")
    # Find g8_data.zip in My Drive or in any folder directly inside it; results are written next to it
    my_drive = Path("/content/drive/MyDrive")
    found = list(my_drive.glob("g8_data.zip")) + list(my_drive.glob("*/g8_data.zip"))
    assert found, "g8_data.zip not found in My Drive - upload it (and wait until the upload has finished)"
    DRIVE_DIR = found[0].parent
    print("Data:", found[0])
    WORK_DIR = Path("/content/g8")                    # fast local disk for the images
except ImportError:                                   # local test run outside Colab
    DRIVE_DIR = Path(os.environ["G8_DRIVE_DIR"])
    WORK_DIR = Path(os.environ["G8_WORK_DIR"])

print("TensorFlow", tf.__version__, "| Keras", keras.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## Settings

`BACKBONE` is the only setting you normally change. Everything else matches the ResNet runs of the submitted
notebook: 384 x 384 images, the same augmentation, dropout 0.5, Adam with weight decay 0.05, label smoothing 0.1,
the 10 metadata numbers (age, sex, body site), 2 epochs for the head and 8 epochs of fine-tuning.

In [ ]:
BACKBONE = "EfficientNetV2S"     # "EfficientNetV2S" or "ConvNeXtTiny"

SMOKE_TEST = os.environ.get("G8_SMOKE") == "1"   # local wiring test only
SEED = 8
IMG_SIZE = 384
BATCH_SIZE = 16
HEAD_EPOCHS = 1 if SMOKE_TEST else 2
FINE_TUNE_EPOCHS = 1 if SMOKE_TEST else 8
TOTAL_EPOCHS = HEAD_EPOCHS + FINE_TUNE_EPOCHS
HEAD_LR = 1e-3
FINE_TUNE_LR = 5e-5
MIN_LR = 1e-6
DROPOUT = 0.5
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
CLASS_NAMES = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
NUM_CLASSES = len(CLASS_NAMES)
SITE_GROUPS = ["torso", "head/neck", "upper extremity", "lower extremity", "palms/soles", "oral/genital"]

# Mixed precision (float16 maths, float32 weights) roughly doubles the speed on NVIDIA GPUs such as the T4
if tf.config.list_physical_devices("GPU") and tf.test.is_built_with_cuda():
    keras.mixed_precision.set_global_policy("mixed_float16")
print("Precision policy:", keras.mixed_precision.global_policy().name)

RUN_DIR = DRIVE_DIR / "results" / f"{BACKBONE}_{IMG_SIZE}{'_smoke' if SMOKE_TEST else ''}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
keras.utils.set_random_seed(SEED)
print("Results:", RUN_DIR)

## Data

The zip holds the images already resized to 384 x 384 and `g8_labels.csv`, which was written by the submitted
notebook: the same image-level HAM10000 split (validation and test only from HAM10000, ISIC 2019 and BCN20000 only in
training), the label index `y`, the metadata and `seen_lesion` (the lesion also has another photo in training).

In [ ]:
image_dir = WORK_DIR / "images"
if not image_dir.is_dir():
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    start = time.time()
    shutil.copy(DRIVE_DIR / "g8_data.zip", WORK_DIR / "g8_data.zip")      # Drive -> local disk is much faster to read
    with zipfile.ZipFile(WORK_DIR / "g8_data.zip") as archive:
        archive.extractall(WORK_DIR)
    (WORK_DIR / "g8_data.zip").unlink()
    print(f"Copied and unpacked in {time.time() - start:.0f} s")

table = pd.read_csv(WORK_DIR / "g8_labels.csv")
table["file"] = [str(image_dir / f"{name}.jpg") for name in table["image"]]
train_df, val_df, test_df = [table[table["split"] == s].reset_index(drop=True) for s in ("train", "val", "test")]
if SMOKE_TEST:
    train_df, val_df, test_df = [d.groupby("label").head(8).reset_index(drop=True) for d in (train_df, val_df, test_df)]
print(f"train {len(train_df):,} | validation {len(val_df):,} | test {len(test_df):,}")


def metadata_matrix(df):
    # One row per image: [age / 100, age unknown, male, female, one column per body-site group]
    age = df["age"].to_numpy(dtype="float32")
    columns = [np.nan_to_num(age / 100), np.isnan(age), df["sex"] == "male", df["sex"] == "female"]
    columns += [df["site"] == group for group in SITE_GROUPS]
    return np.stack([np.asarray(c, dtype="float32") for c in columns], axis=1)


def load_example(file, metadata, label_index):
    image = tf.io.decode_jpeg(tf.io.read_file(file), channels=3)
    image = tf.cast(tf.ensure_shape(image, (IMG_SIZE, IMG_SIZE, 3)), tf.float32)   # pixel values 0 - 255
    return {"image": image, "metadata": metadata}, tf.one_hot(label_index, NUM_CLASSES)


def make_dataset(df, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df["file"].values, metadata_matrix(df), df["y"].values))
    if shuffle:
        ds = ds.shuffle(len(df), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_example, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds, val_ds, test_ds = make_dataset(train_df, shuffle=True), make_dataset(val_df), make_dataset(test_df)

## Model

Same structure as the submitted notebook, only the backbone changes. Both Keras backbones contain their own input
scaling, so they take raw 0 - 255 pixels. BatchNorm layers stay in inference mode (`training=False`) and frozen.

In [ ]:
augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
    keras.layers.RandomRotation(0.25, fill_mode="reflect", seed=SEED),
    keras.layers.RandomZoom(0.2, fill_mode="reflect", seed=SEED),
    keras.layers.RandomTranslation(0.1, 0.1, fill_mode="reflect", seed=SEED),
    keras.layers.RandomBrightness(0.2, value_range=(0, 255), seed=SEED),
    keras.layers.RandomContrast(0.2, value_range=(0, 255), seed=SEED),
], name="augmentation")

base_model = getattr(keras.applications, BACKBONE)(include_top=False, weights="imagenet",
                                                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False

image_input = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
metadata_input = keras.Input(shape=(metadata_matrix(train_df).shape[1],), name="metadata")
x = augmentation(image_input)                               # only active during training
x = base_model(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(DROPOUT)(x)
x = keras.layers.Concatenate()([x, metadata_input])     # 10 metadata numbers next to the image features
logits = keras.layers.Dense(NUM_CLASSES, name="logits")(x)
outputs = keras.layers.Softmax(dtype="float32", name="probabilities")(logits)   # float32 output for a stable loss
model = keras.Model({"image": image_input, "metadata": metadata_input}, outputs, name=f"{BACKBONE}_skin_lesion")
print(f"{BACKBONE}: {base_model.count_params() / 1e6:.1f} M backbone weights")

## Training (stage 1: head only, stage 2: whole backbone except BatchNorm)

In [ ]:
HEAD_WEIGHTS = RUN_DIR / "head.weights.h5"
BEST_WEIGHTS = RUN_DIR / "best.weights.h5"
HISTORY_CSV = RUN_DIR / "training_history.csv"
loss_fn = keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING)


def learning_rate_for(epoch, current_lr=None):
    if epoch < HEAD_EPOCHS:
        return HEAD_LR
    progress = (epoch - HEAD_EPOCHS) / max(FINE_TUNE_EPOCHS - 1, 1)
    return MIN_LR + 0.5 * (FINE_TUNE_LR - MIN_LR) * (1 + math.cos(math.pi * progress))


def read_history():
    if not HISTORY_CSV.exists() or HISTORY_CSV.stat().st_size == 0:
        return pd.DataFrame(columns=["epoch", "accuracy", "loss", "val_accuracy", "val_loss", "learning_rate"])
    return pd.read_csv(HISTORY_CSV).drop_duplicates("epoch", keep="last").sort_values("epoch")


def training_callbacks(backup_name):
    history = read_history()
    best_so_far = history["val_accuracy"].max() if len(history) else None
    return [
        keras.callbacks.LearningRateScheduler(learning_rate_for),
        keras.callbacks.ModelCheckpoint(BEST_WEIGHTS, monitor="val_accuracy", mode="max", save_best_only=True,
                                        save_weights_only=True, initial_value_threshold=best_so_far),
        keras.callbacks.CSVLogger(HISTORY_CSV, append=True),
        keras.callbacks.BackupAndRestore(RUN_DIR / backup_name),
        keras.callbacks.LambdaCallback(on_epoch_begin=lambda epoch, logs: print(
            f"{time.strftime('%H:%M:%S')}  epoch {epoch + 1}/{TOTAL_EPOCHS} started", flush=True)),
    ]


# ---------- Stage 1: train the head only ----------
if HEAD_WEIGHTS.exists():
    model.load_weights(HEAD_WEIGHTS)
    print("Stage 1 already finished")
else:
    model.compile(optimizer=keras.optimizers.Adam(HEAD_LR, weight_decay=WEIGHT_DECAY), loss=loss_fn, metrics=["accuracy"])
    model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, callbacks=training_callbacks("backup_stage1"), verbose=2)
    model.save_weights(HEAD_WEIGHTS)

# ---------- Stage 2: fine-tune the whole backbone (BatchNorm layers stay frozen) ----------
base_model.trainable = True
for layer in base_model.layers:
    layer.trainable = not isinstance(layer, keras.layers.BatchNormalization)
model.compile(optimizer=keras.optimizers.Adam(FINE_TUNE_LR, weight_decay=WEIGHT_DECAY), loss=loss_fn, metrics=["accuracy"])
if len(read_history()) >= TOTAL_EPOCHS:
    print("Stage 2 already finished")
else:
    model.fit(train_ds, validation_data=val_ds, initial_epoch=HEAD_EPOCHS, epochs=TOTAL_EPOCHS,
              callbacks=training_callbacks("backup_stage2"), verbose=2)

history = read_history()
history["epoch"] = history["epoch"] + 1
print(history.round(4).to_string(index=False))

## Curves

In [ ]:
fig, (ax_loss, ax_acc) = plt.subplots(1, 2, figsize=(14, 4.5))
ax_loss.plot(history["epoch"], history["loss"], "o-", label="Training loss")
ax_loss.plot(history["epoch"], history["val_loss"], "o-", label="Validation loss")
ax_lr = ax_loss.twinx()
ax_lr.plot(history["epoch"], history["learning_rate"], "s--", color="tab:green", label="Learning rate")
ax_lr.set_yscale("log")
ax_loss.set_xlabel("Epoch")
ax_loss.legend(loc="upper right")
ax_loss.set_title(f"{BACKBONE}: loss and learning rate")
ax_acc.plot(history["epoch"], history["accuracy"], "o-", label="Training accuracy (with augmentation)")
ax_acc.plot(history["epoch"], history["val_accuracy"], "o-", label="Validation accuracy")
ax_acc.axhline(0.9, color="red", linestyle=":", label="90 % target")
ax_acc.set_xlabel("Epoch")
ax_acc.legend()
ax_acc.set_title(f"{BACKBONE}: accuracy")
plt.savefig(RUN_DIR / "curves.png", dpi=120, bbox_inches="tight")
plt.show()

## Results

The best weights (highest validation accuracy) are evaluated on the training set without augmentation, on the
validation set and once on the test set.

In [ ]:
model.load_weights(BEST_WEIGHTS)
best = history.loc[history["val_accuracy"].idxmax()]
scores = {}
for name, df, ds in [("train (no augmentation)", train_df, make_dataset(train_df)), ("validation", val_df, val_ds),
                     ("test", test_df, test_ds)]:
    probs = model.predict(ds, verbose=0)
    pred, y = probs.argmax(axis=1), df["y"].to_numpy()
    scores[name] = float(np.mean(pred == y))
    if name == "test":
        test_loss = float(loss_fn(keras.utils.to_categorical(y, NUM_CLASSES), probs))
        print(classification_report(y, pred, labels=range(NUM_CLASSES), target_names=CLASS_NAMES, digits=3,
                                    zero_division=0))
    if name != "train (no augmentation)":
        seen = df["seen_lesion"].to_numpy(dtype=bool)
        correct = pred == y
        print(f"{name}: new lesions {correct[~seen].mean():.4f} | lesions also in training {correct[seen].mean():.4f}")

summary = pd.Series({
    "backbone": BACKBONE, "image size": IMG_SIZE, "best epoch": int(best["epoch"]),
    "train accuracy (no augmentation)": round(scores["train (no augmentation)"], 4),
    "best validation accuracy": round(scores["validation"], 4),
    "test accuracy": round(scores["test"], 4), "test loss": round(test_loss, 4),
})
summary.to_csv(RUN_DIR / "summary.csv", header=["value"])
print(summary.to_string())